In [4]:
hi_ensemble.all_states()[0]

Array([0, 0, 1, 1, 0, 0, 1, 1], dtype=int8)

In [7]:
import netket as nk
import jax
import jax.numpy as jnp
import numpy as np

# ================== 系统参数 ==================
n_spatial_orbs = 2
n_alpha = 1
n_beta = 1
n_spin_orbs_per_config = 2 * n_spatial_orbs  # = 4 (α0,α1,β0,β1)

K = 2  # NES-VMC 组态数
n_total_orbs = K * n_spin_orbs_per_config   # 总轨道数 = 8

# ================== 1. 定义 Hilbert 空间 ==================
# 使用 Fock 空间，不添加任何内置约束
hi_ensemble = nk.hilbert.Fock(
    n_max=1,               # 每个轨道最多1个粒子
    N=n_total_orbs,        # 总轨道数
)

# ================== 2. 自定义采样规则 (核心！) ==================
class ConstrainedFermionHopRule(nk.sampler.rules.MetropolisRule):
    """
    自定义规则：在保证每个子组态内粒子数守恒的前提下，进行费米子跃迁。
    """
    def __init__(self, hilbert, K, n_orbs_per_config, edges):
        self.hilbert = hilbert
        self.K = K
        self.n_orbs_per_config = n_orbs_per_config
        self.edges = edges
        self.total_orbs = hilbert.size
        
        # 为每个子组态预计算所有可能的跃迁对 (源轨道, 目标轨道)
        self.all_transitions = []
        for k in range(K):
            offset = k * n_orbs_per_config
            for u, v in edges:
                self.all_transitions.append((offset + u, offset + v))
                self.all_transitions.append((offset + v, offset + u))

        self.all_transitions = jnp.array(self.all_transitions)

    def random_state(self, key, shape):
        """
        生成随机的合法初始状态。
        """
        key = jax.random.PRNGKey(key[0])  # 确保是 PRNGKey 类型
        batch_size = shape[0]
        
        all_configs = []
        for _ in range(batch_size):
            config = jnp.zeros(self.total_orbs, dtype=jnp.int8)
            for k in range(self.K):
                offset = k * self.n_orbs_per_config
                
                # 在每个子组态中，随机选择一个α轨道和一个β轨道来占据
                alpha_orb = offset + jax.random.randint(key, (), 0, 2)
                beta_orb = offset + 2 + jax.random.randint(key, (), 0, 2)
                
                config = config.at[alpha_orb].set(1)
                config = config.at[beta_orb].set(1)
                _, key = jax.random.split(key) # 更新随机种子
            all_configs.append(config)
            
        return jnp.stack(all_configs).reshape(shape)

    def transition(self, key, state):
        """
        提议一个新状态：随机选择一个合法的跃迁并执行。
        """
        state = state[0]  # 取第一条链的状态
        batch_shape = state.shape[:-1]
        state_flat = state.reshape(-1, self.total_orbs)

        # 1. 为批次中的每个状态随机选择一个跃迁对
        trans_idx = jax.random.randint(key, (batch_shape[0],), 0, len(self.all_transitions))
        src = self.all_transitions[trans_idx, 0]
        dst = self.all_transitions[trans_idx, 1]

        # 2. 创建新状态
        def create_new_state(st, s, d):
            # 使用 jnp.where 来向量化地创建新状态
            return st.at[s].set(0).at[d].set(1)

        new_state_flat = jax.vmap(create_new_state)(state_flat, src, dst)

        # 3. 提议对数概率修正项 (对称提议，此项为0)
        log_prob_correction = jnp.zeros(batch_shape[0])

        return new_state_flat.reshape(state.shape), log_prob_correction

    def __repr__(self):
        return f"ConstrainedFermionHopRule(K={self.K})"

# ================== 3. 实例化采样器 ==================
edges_original = [(0, 1), (2, 3)]
custom_rule = ConstrainedFermionHopRule(hi_ensemble, K, n_spin_orbs_per_config, edges_original)

sampler = nk.sampler.MetropolisSampler(
    hi_ensemble,
    rule=custom_rule,
    n_chains=4,
    sweep_size=hi_ensemble.size,
)

# ================== 4. 验证 ==================
# 验证初始状态
initial_state = custom_rule.random_state(jax.random.PRNGKey(42), (1,))
print("初始合法状态 (展平):", initial_state[0])
reshaped = initial_state[0].reshape(K, n_spin_orbs_per_config)
print("子组态 1:", reshaped[0])
print("子组态 2:", reshaped[1])
print("子组态1 α电子数:", jnp.sum(reshaped[0, 0:2]))
print("子组态1 β电子数:", jnp.sum(reshaped[0, 2:4]))

# 验证采样过程
samples = sampler.sample(n_chains=4)
print("\n采样样本形状:", samples.shape)
sample0 = samples[0]
reshaped_sample = sample0.reshape(K, n_spin_orbs_per_config)
print("采样得到的子组态 1:", reshaped_sample[0])
print("采样得到的子组态 2:", reshaped_sample[1])

NetKetPyTreeUndeclaredAttributeAssignmentError: 
            Tried to assign an undeclared attribute "hilbert" to the Pytree class "__main__.ConstrainedFermionHopRule" during
            the __init__ method.

            Declared attributes for "__main__.ConstrainedFermionHopRule" are "('_pytree__node_fields',)".

            This error is thrown when you try to assign an attribute to a NetKet-style Pytree ( a class that
            inherits from `nk.utils.struct.Pytree`) that was not declared in the class definition.

            To fix this error, simply declare the attributes in the class definition as shown in the example below:

                from netket.utils import struct
                import jax

                class MyPytree(struct.Pytree):
                    my_dynamic_attribute: jax.Array
                    my_static_attribute : int = struct.field(pytree_node=False)

                    def __init__(self, dyn_val, static_val):
                        self.my_dynamic_a

In [ ]:
from flax import nnx
import jax.numpy as jnp

#==============================================================================
# 3. 神经网络模型（正确版，无 log 污染 M 矩阵）
#==============================================================================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)  # ✅ 输出原始 ψ，不是 log！

class NESTotalAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals, n_states=2, hidden_dim=8, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_states = n_states
        self.n_spin_orbitals = n_spin_orbitals
        self.single_ansatz_list = [
            SingleStateAnsatz(n_spin_orbitals, hidden_dim, rngs=nnx.Rngs(42+i))
            for i in range(n_states)
        ]

    def __call__(self, x_batch):
        K = self.n_states
        M = []
        x_batch = jnp.array(x_batch).reshape(K, -1)
        # ✅ 构建原始波函数矩阵 M_ij = ψ_j(x^i)
        for i in range(K):
            row = [self.single_ansatz_list[j](x_batch[i]) for j in range(K)]
            M.append(jnp.array(row))
        M = jnp.stack(M)  # shape = (K, K)

        # ✅ 总波函数 = det(M)，绝对不能把 log 放进 M！
        psi_total = jnp.linalg.det(M)

        # ✅ NetKet 采样只需要 log_psi_total
        log_psi_total = jnp.log(psi_total)

        # ✅ 返回：logψ（给采样器）、det值、原始M矩阵（给能量计算）
        return log_psi_total,M

In [ ]:
# 初始化模型
rngs = nnx.Rngs(42)
K=3
total_ansatz = NESTotalAnsatz(4, n_states=K, hidden_dim=16, rngs=rngs)

#==============================================================================
# 4. 核心：哈密顿作用 + 局部能量矩阵（论文标准）
#==============================================================================
def Ham_psi(ha: nk.operator.DiscreteOperator, single_ansatz, x: jnp.array):
    x_primes, mels = ha.get_conn(x)
    psi_vals = jax.vmap(single_ansatz)(x_primes)
    return jnp.sum(mels * psi_vals)

def Ham_Psi(ha:nk.operator.DiscreteOperator, total_ansatz: NESTotalAnsatz, x_batch):
    K_state = total_ansatz.n_states
    H_mat = []
    for i in range(K_state):
        row = []
        for j in range(K_state):
            v = Ham_psi(ha, total_ansatz.single_ansatz_list[j], x_batch[i])
            row.append(v)
        H_mat.append(row)
    return jnp.array(H_mat, dtype=complex)

def compute_local_energy(ha:nk.operator.DiscreteOperator, total_ansatz: NESTotalAnsatz, x_batch, return_log_psi=False):
    psi, M = total_ansatz(x_batch, return_log_psi=return_log_psi)
    # 🔥 修复 1：对角加载，防止矩阵奇异
    eps = 1e-3
    M = M + eps * jnp.eye(M.shape[0], dtype=M.dtype)
    Hp = Ham_Psi(ha, total_ansatz, x_batch)
    el_mat = jnp.linalg.solve(M, Hp)
    return jnp.trace(el_mat), el_mat

#==============================================================================
# 5. 损失函数 & 训练步骤（JAX 自动求导）
#==============================================================================
@jax.jit
def loss_fn(model_state, samples):
    nnx.update(total_ansatz, model_state)
    total_energy = 0.0 + 0j
    n_samples = samples.shape[0]
    for xb in samples:
        tr_EL, _ = compute_local_energy(ha, total_ansatz, xb)
        total_energy += tr_EL
    avg_energy = total_energy.real / n_samples
    return avg_energy

@jax.jit
def train_step(model_state, samples):
    loss, grads = jax.value_and_grad(loss_fn)(model_state, samples)
    model_state = jax.tree_util.tree_map(lambda p, g: p - 0.01 * g, model_state, grads)
    return model_state, loss

In [ ]:
hi_ensemble.all_states()[0].reshape(3,-1)

In [ ]:
# 完全符合论文：采样 |TotalAnsatz|²
def forward(state, x_batch):
    # 正确解包：log_psi_total 是数值，new_state 是状态
    (log_psi_total,M), new_state = nnx.call(state)(x_batch)
    # 直接用 数值 log_psi_total 填充，完美兼容 jnp.full
    return jnp.full((K,), log_psi_total)

parameters = nnx.split(total_ansatz)

sampler_state = sampler.init_state(forward, parameters, seed=1)
#sampler_state = sampler.reset(forward, parameters, sampler_state)

samples, sampler_state = sampler.sample(
    forward, parameters, state=sampler_state, chain_length=500
)
#samples.shape = (K,batch_size,n_spin_orbitals) = (2,500,4)

In [ ]:
x_batch = jnp.array([[1,0,1,0],[0,1,0,1]])
log_psi,M = total_ansatz(x_batch)

In [ ]:
jnp.log(jnp.linalg.det(M))


In [ ]:
parameters = nnx.split(total_ansatz)
(a,b),m = nnx.call(parameters)(x_batch)